### **Section 2: Dask Local**

#### **Timing at ≥4096×4096 and speedup vs Numba single-core**

In [1]:
import numpy as np
# A copy from lecture 6, milestone 1

# A copy from lecture 5

from numba import njit

# From slide 27, with the implementation filled in by me
# based on lecture 4.

# From L04 (unchanged — add `cache=True`` if not already there):
@njit(cache=True)
def mandelbrot_pixel(c_real, c_imag, max_iter):
    z_real = z_imag = 0.0
    for i in range(max_iter):
        zr2 = z_real*z_real
        zi2 = z_imag*z_imag
        if zr2 + zi2 > 4.0: return i
        z_imag = 2.0*z_real*z_imag + c_imag
        z_real = zr2 - zi2 + c_real
    return max_iter

@njit(cache=True)
def mandelbrot_chunk(row_start, row_end, N, x_min, x_max, y_min, y_max, max_iter):
    out = np.empty((row_end - row_start, N), dtype=np.int32)
    dx = (x_max - x_min) / N
    dy = (y_max - y_min) / N
    for r in range(row_end - row_start):
        c_imag = y_min + (r + row_start) * dy
        for col in range(N):
            out[r, col] = mandelbrot_pixel(x_min + col*dx, c_imag, max_iter)
    return out

In [2]:
# Mostly a copy from Lecture 6, Milestone 1

# From slide 34

from dask import delayed
from dask.distributed import Client, LocalCluster
import dask, numpy as np, time, statistics
# mandelbrot_chunk: your @njit(cache=True) function from L04/L05
def mandelbrot_dask(N, x_min, x_max, y_min, y_max,
                    max_iter=100, n_chunks=32):
    chunk_size = max(1, N // n_chunks)
    tasks, row = [], 0
    while row < N:
        row_end = min(row + chunk_size, N)
        tasks.append(delayed(mandelbrot_chunk)(
            row, row_end, N, x_min, x_max, y_min, y_max, max_iter))
        row = row_end
    parts = dask.compute(*tasks)
    return np.vstack(parts)

In [3]:
# Mostly a copy of my own code from Lecture 6, Milestone 2
# but using 4096x4096 instead excluding chunk sweep.

def compute_mp2_section2():
    n_sizes = [512, 1024, 2048, 4096]
    max_iter = 100
    X_MIN, X_MAX, Y_MIN, Y_MAX = -2.5, 1.0, -1.25, 1.25
    cluster = LocalCluster(n_workers=8, threads_per_worker=1)
    client = Client(cluster)
    client.run(lambda: mandelbrot_chunk(0, 8, 8, X_MIN, X_MAX,  # warm up all workers
                                        Y_MIN, Y_MAX, 10))

    times_res = []

    for n in n_sizes:
        times = []
        for _ in range(3):
            t0 = time.perf_counter()
            result = mandelbrot_dask(
                n, X_MIN, X_MAX, Y_MIN, Y_MAX, max_iter)
            times.append(time.perf_counter() - t0)

        T_p = statistics.median(times)
        times_res.append(T_p)

    client.close()
    cluster.close()

    return n_sizes, times_res

In [4]:
from pathlib import Path
import pandas as pd

mp2_section2 = Path("mp2_section2.csv")

if not mp2_section2.exists():
    n_sizes, times, = compute_mp2_section2()
    df = pd.DataFrame({
        "n_size": n_sizes, 
        "time": times,
    })
    df.to_csv(mp2_section2, index_label=False)

In [5]:
from IPython.display import Markdown, display

mp2_section2 = pd.read_csv(mp2_section2)

t0 = 0.71728592099862  # serial numba result from lecture 7
mp2_section2["speedup"] = t0 / mp2_section2["time"]


mp2_section2["n_size"] = mp2_section2["n_size"].map("{:>2}".format)
mp2_section2["time"] = mp2_section2["time"].map("{:>.3f}s".format)
mp2_section2["speedup"] = mp2_section2["speedup"].map("{:>.2f}x".format)

display(Markdown(mp2_section2.to_markdown(index=False)))

|   n_size | time   | speedup   |
|---------:|:-------|:----------|
|      512 | 0.052s | 13.70x    |
|     1024 | 0.059s | 12.10x    |
|     2048 | 0.088s | 8.13x     |
|     4096 | 0.266s | 2.70x     |